# Fabric Benchmarking

## Data Source

The use case is implmented using open data provided by the [UK Land Registry House Price Data open data repository](https://www.gov.uk/government/statistical-data-sets/price-paid-data-downloads).

This data is made available for us under an [Open Government Licence](https://www.nationalarchives.gov.uk/doc/open-government-licence/version/3/).

The data is provided as a set of CSV files, one for year calendar year, which have been downloaded ont a Farbric lakehouse.

The data has been collected since 1995, with circa 1 million property sales per year on average, all 30 years of historic data is ~30 million rows for data and ~5GB of raw CSV data.

This is a typical dataset that we encounter for common enterprise client use cases.  The data we are working with will fit in memory for all of the platform configurations we will be testing.  We find that many benchmarks focus on processing 

The objective is focus on the constraints we more often find we are working with:

- Developer exerperience - having processes that run rapidly unlocks significant benefits during the development phase: test suites run quicker, the inner development loop is optimised, time to value is accelerated.  In today's rapidly evolving environment this can yield significant advantages.

- TCO - the cost of running data pipelines both in terms of financial and environmental impact is becoming a significant factor for many organisations.

## Use Case

The use case mimics a common set of data transformations that you would see on data of this nature.  It includes:

0. Start Up & Set Up - the overhead of provisioning the platform (Spark or Python) on whcih the code is going to run, then completing various tasks such as importing Python packages.
1. Ingestion & Transformation - reading raw data from a set of CSV files, standardising, cleaning and adding new features.
2. Create Dimensional Model - taking different slices of the transformed data and writing that out to the lakehouse in Delta format  data for downstream consumption, in this case as a dimensional model for Power BI.
3. Reading and Summarise - reading the tables back in running analysis based on filtering, joining and summarising the data across different categories.
4. Capture & Clean Up - at various points in the process above a set of benchmark timestamps are captured along with other metadata such as memory consumption.  These are written to permanent storage in the lakehouse for analysis.

A more detailed overview of this process is captured in the mermaid diagram below, with the following symbology:
- ⬆️ - reading from lakehouse.
- 🔧 - data wrangling.
- ⬇️ - write to lakehouse.
- 📊 - points in the process where a benchmark timestamp is captured.


```mermaid
flowchart LR

subgraph Phase0["Phase 0 - Start Up & Set Up"]
    direction TB
    A1[Start Up Platform] --> A2[Import Packages]
    A2 --> A3[Set Up Logging]
    A3 --> A4[Define Constants]
    A4 --> A5[Set Up Helper Functions]
    A5 --> A6[Configure Paths]
    A6 --> A7[Initialise BenchmarkManager]
    A7 --> A8["📊 capture: setup"]
end

subgraph Phase1["Phase 1 - Ingest & Transform"]
    direction TB
    B1["⬆️Scan CSV Files"] --> B2["📊 capture: ingest"]
    B2 --> B3["🔧Transform Data"]
    B3 --> B4["Cache Transformed Data"]
    B4 --> B5["📊 capture: transform"]
end

subgraph Phase2["Phase 2 - Create Dimensional Model"]
    direction TB
    C1[🔧Create Prices Table] --> C2[⬇️Write Prices to Delta]
    C2 --> C3["📊 capture: write_prices"]
    
    C3 --> D1["🔧Create Dates Dimension"]
    D1 --> D2["⬇️Write Dates to Delta"]
    D2 --> D3["📊 capture: write_dates"]
    
    D3 --> E1["🔧Create Locations Dimension"]
    E1 --> E2["⬇️Write Locations to Delta"]
    E2 --> E3["📊 capture: write_locations"]
end

subgraph Phase3["Phase 3 - Read & Summarise"]
    direction TB
    F1[⬆️Read Prices from Delta] --> F2["📊 capture: read_prices"]
    F2 --> F3["⬆️Read Dates from Delta"]
    F3 --> F4["📊 capture: read_dates"]
    F4 --> F5["🔧Join Prices ⟕ Dates"]
    F5 --> F6["🔧Aggregate by Month & Property Type"]
    F6 --> F7["🔧Collect & Display Results"]
    F7 --> F8["📊 capture: join_and_summarise"]
end

subgraph Phase4["Phase 4 - Capture & Clean Up"]
    direction TB
    G1[⬇️Export Benchmarks] --> G2[Calculate Elapsed Time]
    G2 --> G3[Remove Working Data]
end

Phase0 --> Phase1
Phase1 --> Phase2
Phase2 --> Phase3
Phase3 --> Phase4

%% subgraph DataFlow["Data Flow"]
%%     direction LR
%%     CSV[(CSV Files<br/>Land Registry)] -.->|read| B1
%%     B4 -.->|source| C1
%%     B4 -.->|source| D1
%%     B4 -.->|source| E1
%%     C2 -.->|write| Delta1[(prices<br/>Delta Table)]
%%     D2 -.->|write| Delta2[(dates<br/>Delta Table)]
%%     E2 -.->|write| Delta3[(locations<br/>Delta Table)]
%%     Delta1 -.->|read| F1
%%     Delta2 -.->|read| F3
%% end
```

## Workloads

This study was carried out to compare running the common use case implemented across 4 different workloads running on the Fabric Platform:

1. Pandas - the default package for those who first introduced data engineering using Python.  The package has a huge following.  Only suitable for data volumes which can fit into memory.

2. PySpark - the Python API for Apache Spark, a common choice for enterprise platform for data engineering.  Made popular by Databricks and Azure Synapse which provide Spark as cloud PaaS.  Spark is a distributed compute platform which can scale up to handle true "big data" workloads.

3. Polars - a Rust engine with a Python API which provides a powerful query engine with a dataframe based API.  See blog post X for more details.

4. DuckDB - a C++ engine with a Python API which provides an analytical database engine.  See blog post Y for more details.

## Platforms

Fabric offers mutliple compute platforms.  For this study we leveraged two:

- Spark notebooks - a notebook experience over a Spark cluster hosted on Fabric.  Enables polyglot development (Python, R, SQL) over a Spark cluster which is spun up according to your chosen configuration (vCores, memory, number of executor nodes) on demand.

- Python notebooks - a relatively new addition to Fabric.  Python notebooks provide a single node for execution which can be sized according to range of pre-defined configurations (vCores and memory).  Whilst they are designed for "smaller" workloads, we find that the majority of enterprise use cases can be accomodated on this platform through intelligent choice of tooling and design.


## Configurations

It is difficult to achieve parity across the Spark and Python notebook platforms.  We opted for the following configurations which we labelled using "T Shirt Sizes" for ease of cross comparison:

| T-Shirt Size | Python Notebook Configuration | Spark Pool Configuration |
| --- | ---                 | --- |
| XS  | 4 vCores, 16G RAM   |  |
| S   | 8 vCores, 32G RAM   | 1 Executor 4/4 vCores 28G/28G RAM |
| M   | 16 vCores, 64G RAM  | 1 Executor 8/8 vCores 56G/56G RAM<br>2 Executors 4/4 vCores 28G/28G RAM |
| L   | 32 vCores, 128G RAM | 2 Executors 8/8 vCores 56G/56G RAM<br>4 Executors 4/4 cores 28G/28G RAM |
| XL  | 64 vCores, 256G RAM | 4 executors 8/8 vCores 56G/56G RAM |

## Methodology

Multiple runs were completed for each combination of: Platform, Workload and Configuration to enable median times to be captured.

## Analysis

In [4]:
import polars as pl
from azure.identity import InteractiveBrowserCredential
import plotly.express as px

In [5]:
credential = InteractiveBrowserCredential()

In [6]:
token = credential.get_token("https://storage.azure.com/.default")

In [7]:
# Pre-requisities are to create a Fabric Workspace with a lakehouse, putting names here:
WORKSPACE_NAME = "fabric_performance_benchmark_workspace"
LAKEHOUSE_NAME = "fabric_performance_benchmark_lakehouse"

In [8]:
# Helper function to create base ABFSS path based on workspace and lakehouse name
def construct_base_abfss_path(workspace_name: str, lakehouse_name: str) -> str:
    """Construct the base ABFSS path for a given workspace and lakehouse."""
    # Because it is a URL, replace spaces with %20
    workspace_name = workspace_name.replace(" ", "%20")
    lakehouse_name = lakehouse_name.replace(" ", "%20")
    return f"abfss://{workspace_name}@onelake.dfs.fabric.microsoft.com/{lakehouse_name}.Lakehouse"

# Helper function to create storage options that enable data tools to authenticate and interact with onelake storage
def create_storage_options() -> dict:
    return {
        "bearer_token": token.token,
        "use_fabric_endpoint": "true"
    }

In [9]:
benchmarks_path = f"{construct_base_abfss_path(WORKSPACE_NAME, LAKEHOUSE_NAME)}/Tables/benchmark_repository/benchmarks"

stages_path = f"{construct_base_abfss_path(WORKSPACE_NAME, LAKEHOUSE_NAME)}/Tables/benchmark_repository/stages"

configurations_path = f"{construct_base_abfss_path(WORKSPACE_NAME, LAKEHOUSE_NAME)}/Tables/benchmark_repository/configurations"

benchmark_analytics_path = f"{construct_base_abfss_path(WORKSPACE_NAME, LAKEHOUSE_NAME)}/Tables/benchmark_repository/benchmark_analytics"

In [10]:
storage_options = create_storage_options()

## Load Benchmark Data For Analysis

In [11]:
# Load prices from and filter them to exclude "Other" property types
benchmarks = pl.read_delta(benchmark_analytics_path, storage_options=storage_options)

In [12]:
benchmarks

order,platform,configuration,workload_name,run_timestamp,stage_name,stage_time,cpu_count,cpu_usage,memory,memory_usage,stage_time_delta,stage_order,phase,phase_order,configuration_scale,t_shirt_size,cumulative_time
i32,str,str,str,str,str,datetime[μs],i64,f64,f64,f64,f64,i64,str,i64,str,str,f64
1,"""Fabric Python Notebook""","""02 vCores""","""duckdb_benchmark""","""20260203_180137""","""start""",2026-02-03 18:01:37,2,6.1,15.36681,14.3,null,1,"""start_and_setup""",1,"""10""","""XS""",null
2,"""Fabric Python Notebook""","""02 vCores""","""duckdb_benchmark""","""20260203_180137""","""setup""",2026-02-03 18:02:02.346593,2,7.8,15.36681,14.3,25.346,2,"""start_and_setup""",1,"""10""","""XS""",25.346
3,"""Fabric Python Notebook""","""02 vCores""","""duckdb_benchmark""","""20260203_180137""","""ingest""",2026-02-03 18:02:05.585916,2,52.5,15.36681,14.8,3.239,3,"""ingest_and_transform""",2,"""10""","""XS""",28.585
4,"""Fabric Python Notebook""","""02 vCores""","""duckdb_benchmark""","""20260203_180137""","""transform""",2026-02-03 18:03:12.638043,2,85.7,15.36681,56.9,67.052,4,"""ingest_and_transform""",2,"""10""","""XS""",95.637
5,"""Fabric Python Notebook""","""02 vCores""","""duckdb_benchmark""","""20260203_180137""","""write_prices""",2026-02-03 18:03:30.211017,2,39.8,15.36681,59.8,17.572,5,"""create_and_write""",3,"""10""","""XS""",113.209
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
6,"""Fabric PySpark Notebook""","""04 executors 08/08 cores 56g/5…","""pyspark_benchmark""","""20260205_181302""","""write_dates""",2026-02-05 18:17:42.617234,8,58.4,62.545181,30.1,4.708,7,"""create_and_write""",3,"""55""","""XL""",280.616
7,"""Fabric PySpark Notebook""","""04 executors 08/08 cores 56g/5…","""pyspark_benchmark""","""20260205_181302""","""write_locations""",2026-02-05 18:17:57.730503,8,44.5,62.545181,30.3,15.113,6,"""create_and_write""",3,"""55""","""XL""",295.729
8,"""Fabric PySpark Notebook""","""04 executors 08/08 cores 56g/5…","""pyspark_benchmark""","""20260205_181302""","""read_prices""",2026-02-05 18:17:58.619615,8,28.4,62.545181,30.3,0.889,8,"""read_and_summarise""",4,"""55""","""XL""",296.618


## Overall Results

In [13]:
overall_benchmarks = (
    benchmarks
    .filter(pl.col("phase") != "start_and_setup")
    .sort(["run_timestamp", "order"])
    .group_by(["platform", "configuration", "workload_name", "run_timestamp", "t_shirt_size"])
    .agg(pl.col("stage_time_delta").sum().alias("total_time"))
    .sort(["platform", "configuration", "workload_name", "run_timestamp"])
)

In [14]:
fig = px.box(overall_benchmarks, 
             x="t_shirt_size",
             y="total_time",
             color="workload_name",
             title="Overview Of Results",
             category_orders={
                 "t_shirt_size": ["XS", "S", "M", "L", "XL"],
                 "workload_name": ["pandas_benchmark", "pyspark_benchmark", "polars_benchmark", "duckdb_benchmark"]
                 }
             )

fig.add_vline(x=0.5, line_width=1, line_color="white")
fig.add_vline(x=1.5, line_width=1, line_color="white")
fig.add_vline(x=2.5, line_width=1, line_color="white")
fig.add_vline(x=3.5, line_width=1, line_color="white")


fig.show()

## Stage Analysis

In [23]:
stage_cumulative_time = (
    benchmarks
    .filter(pl.col("phase") != "start_and_setup")
    .sort(["run_timestamp", "order"])
    .group_by(["platform", "configuration", "workload_name", "t_shirt_size", "stage_name", "order"])
    .agg(pl.col("cumulative_time").median().alias("median_cumulative_time"))
    .sort(["platform", "configuration", "workload_name", "t_shirt_size", "order"])
)  

In [25]:
fig = px.line(
    stage_cumulative_time,
    x="stage_name", 
    y="median_cumulative_time",
    color="workload_name",  # Different line color per workload_name
    facet_col="t_shirt_size",  # Different dash pattern per t_shirt_size
    markers=True,
    title="Median Cumulative Time by Stage and T-Shirt Size",
    category_orders={
                 "t_shirt_size": ["XS", "S", "M", "L", "XL"],
                 "workload_name": ["pandas_benchmark", "pyspark_benchmark", "polars_benchmark", "duckdb_benchmark"],
                 "stage_name": ["ingest", "transform", "write_prices", "write_dates", "write_locations", "read_prices", "read_dates", "join_and_summarise"]
                 }
)
fig.show()